In [28]:
from utils import read_video, save_video
from trackers import Tracker
import cv2
import copy
import numpy as np
import pandas as pd
from team_assigner import TeamAssigner
from player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator import CameraMovementEstimator
from view_transformer import ViewTransformer
from speed_and_distance_estimator import SpeedAndDistance_Estimator

In [29]:
# Inicializar video
video_frames = read_video('input_videos/08fd33_4.mp4')
print(f"Vídeo cargado. Número total de frames: {len(video_frames)}")

# Crear un array de relleno con 'unos' para evitar la división por cero.
placeholder_ball_control = np.ones(len(video_frames), dtype=int)

Vídeo cargado. Número total de frames: 750


In [31]:
# Paso 1: Tracking
# Inicializar Tracker
tracker = Tracker('models/best_1.pt')
tracks = tracker.get_object_tracks(video_frames,
                                   read_from_stub=True,
                                   stub_path='stubs/track_stubs.pkl')

# Añadir la posición a los tracks
tracker.add_position_to_tracks(tracks)

In [32]:
# Paso 2: Estimar movimiento de la cámara
camera_movement_estimator = CameraMovementEstimator(video_frames[0])
camera_movement_per_frame = camera_movement_estimator.get_camera_movement(video_frames,
                                                                          read_from_stub=True,
                                                                          stub_path='stubs/camera_movement_stub.pkl')
# Ajustar las posiciones de los tracks según el movimiento de la cámara
camera_movement_estimator.add_adjust_positions_to_tracks(tracks, camera_movement_per_frame)

In [33]:
# Paso 3: Transformación de perspectiva
view_transformer = ViewTransformer()
view_transformer.add_transformed_position_to_tracks(tracks)
print("Transformación de perspectiva aplicada. Las coordenadas 2D han sido añadidas a la estructura 'tracks'.")
print("-" * 50)

# Preparar y mostrar los datos en una tabla para un frame específico
frame_to_visualize = 4
player_positions = tracks['players'][frame_to_visualize]

# Creamos una lista para almacenar los datos de cada jugador
data_for_table = []

# Iteramos sobre cada jugador detectado en el frame
for track_id, player_data in player_positions.items():
    # Extraemos la posición original (en píxeles) y la transformada
    original_pos = player_data.get('position', 'N/A')
    transformed_pos = player_data.get('transformed_position', 'N/A')
    
    # Añadimos los datos a nuestra lista
    data_for_table.append({
        "ID del Jugador": track_id,
        "Posición Original (px)": original_pos,
        "Posición Transformada (campo)": transformed_pos
    })

# Creamos el DataFrame de Pandas si tenemos datos
if data_for_table:
    df = pd.DataFrame(data_for_table)
    
    # Configuramos Pandas para mostrar toda la información
    pd.set_option('display.max_colwidth', None)
    
    print(f"Mostrando la comparación de coordenadas para el Frame #{frame_to_visualize}:\n")
    display(df)
else:
    print(f"No se encontraron datos de jugadores para mostrar en el frame {frame_to_visualize}.")

Transformación de perspectiva aplicada. Las coordenadas 2D han sido añadidas a la estructura 'tracks'.
--------------------------------------------------
Mostrando la comparación de coordenadas para el Frame #4:



,ID del Jugador,Posición Original (px),Posición Transformada (campo)
0,7,"(1014, 528)",N/A
1,3,"(350, 568)",N/A
2,13,"(1602, 708)",N/A
3,8,"(896, 423)",N/A
4,4,"(1337, 516)",N/A
5,9,"(1251, 501)",N/A
6,14,"(557, 785)",N/A
7,1,"(897, 723)",N/A
8,6,"(1395, 905)",N/A
9,11,"(377, 832)",N/A


In [34]:
# Paso 4: Interpolar posiciones del balón
tracks["ball"] = tracker.interpolate_ball_positions(tracks["ball"])

In [35]:
# Paso 5: Cálculo de velocidad y distancia
speed_and_distance_estimator = SpeedAndDistance_Estimator()
speed_and_distance_estimator.add_speed_and_distance_to_tracks(tracks)
print("Cálculo de velocidad y distancia completado. Los datos han sido añadidos a la estructura 'tracks'.")
print("-" * 50)


# Preparar y mostrar los datos en una tabla para un frame específico
frame_to_visualize = 100 

# Asegurarnos de que el frame elegido no se salga del rango del vídeo
if frame_to_visualize >= len(tracks['players']):
    frame_to_visualize = len(tracks['players']) - 1

player_data_for_frame = tracks['players'][frame_to_visualize]

# Creamos una lista para almacenar los datos de la tabla
data_for_table = []

# Iteramos sobre cada jugador detectado en ese frame
for track_id, player_data in player_data_for_frame.items():
    # Extraemos la velocidad y la distancia calculadas.
    speed = player_data.get('speed', 0.0)
    distance = player_data.get('distance', 0.0)
    
    data_for_table.append({
        "ID del Jugador": track_id,
        "Velocidad (km/h)": f"{speed:.2f}", 
        "Distancia Recorrida (m)": f"{distance:.2f}"
    })

# Creamos y mostramos el DataFrame de Pandas
if data_for_table:
    df = pd.DataFrame(data_for_table)
    print(f"Mostrando Velocidad y Distancia para el Frame #{frame_to_visualize}:\n")
    display(df)
else:
    print(f"No se encontraron datos de jugadores para mostrar en el frame {frame_to_visualize}.")

print("-" * 50)

Cálculo de velocidad y distancia completado. Los datos han sido añadidos a la estructura 'tracks'.
--------------------------------------------------
Mostrando Velocidad y Distancia para el Frame #100:



,ID del Jugador,Velocidad (km/h),Distancia Recorrida (m)
0,11,39.97,27.15
1,13,0.00,0.00
2,34,71.65,41.08
3,1,54.99,30.32
4,5,54.11,35.83
5,19,0.00,0.00
6,14,20.24,28.64
7,22,0.00,0.00
8,9,0.00,0.00
9,15,44.84,25.24


--------------------------------------------------


In [36]:
# Paso 6: Asignar equipo a cada jugador
team_assigner = TeamAssigner()
# Se analiza el primer frame para determinar los colores de los dos equipos
team_assigner.assign_team_color(video_frames[0], 
                                tracks['players'][0])

# Iteramos por cada frame para asignar el equipo a cada jugador detectado
for frame_num, player_track in enumerate(tracks['players']):
    for player_id, track in player_track.items():
        team = team_assigner.get_player_team(video_frames[frame_num],   
                                             track['bbox'],
                                             player_id)
        # Guardamos el equipo y el color del equipo en nuestra estructura de datos 'tracks'
        tracks['players'][frame_num][player_id]['team'] = team 
        tracks['players'][frame_num][player_id]['team_color'] = team_assigner.team_colors[team]

In [37]:
# Paso 7: Asignar posesión del balón
player_assigner = PlayerBallAssigner()
# Creamos la variable real. Ya no usaremos el placeholder.
team_ball_control = [] 
for frame_num, player_track in enumerate(tracks['players']):
    ball_bbox = tracks['ball'][frame_num][1]['bbox']
    assigned_player = player_assigner.assign_ball_to_player(player_track, ball_bbox)

    if assigned_player != -1:
        # Si un jugador tiene el balón, marcamos que lo tiene y registramos a su equipo
        tracks['players'][frame_num][assigned_player]['has_ball'] = True
        team_ball_control.append(tracks['players'][frame_num][assigned_player]['team'])
    else:
        # Si nadie tiene el balón, asumimos que el último equipo en tocarlo mantiene la posesión
        if team_ball_control:
            team_ball_control.append(team_ball_control[-1])
        else:
            # Para el caso inicial donde nadie tiene el balón al principio
            team_ball_control.append(0)

# Convertimos la lista a un array de NumPy para que las funciones de tracker.py funcionen
team_ball_control = np.array(team_ball_control)

# Dibujar el vídeo final con TODAS las anotaciones
print("Generando el vídeo final con todas las anotaciones...")

# Usamos la variable 'team_ball_control' que acabamos de calcular
# output_video_frames_final = tracker.draw_annotations(video_frames, tracks, team_ball_control)

# Añadimos las otras capas de información visual
# output_video_frames_final = camera_movement_estimator.draw_camera_movement(output_video_frames_final, camera_movement_per_frame)
# output_video_frames_final = speed_and_distance_estimator.draw_speed_and_distance(output_video_frames_final, tracks)

# Guardar el vídeo final
# save_video(output_video_frames_final, 'output_videos/output_video_final.avi')
# print("Vídeo final con todas las anotaciones guardado en 'output_videos/output_video_final.avi'")

Generando el vídeo final con todas las anotaciones...


In [41]:
type(tracks)

dict

In [40]:
print(tracks)

{'players': [{1: {'bbox': [851.27490234375, 632.7650756835938, 900.3116455078125, 721.5734252929688], 'position': (875, 721), 'position_adjusted': (875, 721), 'position_transformed': [58.41587448120117, 55.6447868347168], 'speed': 43.66306736387633, 'distance': 2.5267978798539543, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}, 3: {'bbox': [328.891357421875, 494.5967102050781, 364.2227478027344, 568.6959838867188], 'position': (346, 568), 'position_adjusted': (346, 568), 'position_transformed': [13.878900527954102, 42.18819046020508], 'speed': 8.548121424949201, 'distance': 0.49468295283270836, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}, 4: {'bbox': [1309.0023193359375, 446.935546875, 1349.9012451171875, 516.509033203125], 'position': (1329, 516), 'position_adjusted': (1329, 516), 'position_transformed': None, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}, 5: {'bbox': [587.555419921875, 589.6014404296875

In [42]:
len(tracks)

3

In [45]:
tracks['players'][0].keys()

dict_keys([1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 22])

In [52]:
tracks['players'][0].values()

dict_values([{'bbox': [851.27490234375, 632.7650756835938, 900.3116455078125, 721.5734252929688], 'position': (875, 721), 'position_adjusted': (875, 721), 'position_transformed': [58.41587448120117, 55.6447868347168], 'speed': 43.66306736387633, 'distance': 2.5267978798539543, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}, {'bbox': [328.891357421875, 494.5967102050781, 364.2227478027344, 568.6959838867188], 'position': (346, 568), 'position_adjusted': (346, 568), 'position_transformed': [13.878900527954102, 42.18819046020508], 'speed': 8.548121424949201, 'distance': 0.49468295283270836, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}, {'bbox': [1309.0023193359375, 446.935546875, 1349.9012451171875, 516.509033203125], 'position': (1329, 516), 'position_adjusted': (1329, 516), 'position_transformed': None, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}, {'bbox': [587.555419921875, 589.6014404296875, 630.6329956

In [53]:
tracks['players'][0].items()

dict_items([(1, {'bbox': [851.27490234375, 632.7650756835938, 900.3116455078125, 721.5734252929688], 'position': (875, 721), 'position_adjusted': (875, 721), 'position_transformed': [58.41587448120117, 55.6447868347168], 'speed': 43.66306736387633, 'distance': 2.5267978798539543, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}), (3, {'bbox': [328.891357421875, 494.5967102050781, 364.2227478027344, 568.6959838867188], 'position': (346, 568), 'position_adjusted': (346, 568), 'position_transformed': [13.878900527954102, 42.18819046020508], 'speed': 8.548121424949201, 'distance': 0.49468295283270836, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}), (4, {'bbox': [1309.0023193359375, 446.935546875, 1349.9012451171875, 516.509033203125], 'position': (1329, 516), 'position_adjusted': (1329, 516), 'position_transformed': None, 'team': 1, 'team_color': array([     141.15,      228.08,      181.77])}), (5, {'bbox': [587.555419921875, 589.60144042

In [54]:
tracks['players'][0][1].keys()

dict_keys(['bbox', 'position', 'position_adjusted', 'position_transformed', 'speed', 'distance', 'team', 'team_color'])